In [0]:
def list_tables_in_schema(catalog: str, schema: str) -> list:
    """
    Returns a list of all table names in the specified catalog and schema.
    Parameters:
    - catalog (str): The catalog name (e.g., 'hive_metastore' or 'your_catalog_name').
    - schema (str): The schema/database name within the catalog.
    Returns:
    - list: A list of table names as strings.
    """
    tables_list = []
    tables_dict = {}
    try:
        full_schema = f"{catalog}.{schema}"
        tables = spark.catalog.listTables(full_schema)
        for table in tables:
            tables_list.append(table.name)
            tables_dict[table.name] = table.description
        return tables_list, tables_dict
    except Exception as e:
        print(f"Error listing tables in {catalog}.{schema}: {e}")
        return None, None

# Unit test
# list_tables_in_schema("hive_metastore", "default")

In [0]:
def get_column_comments(catalog: str, schema: str, table: str) -> dict:
    """
    Returns a dictionary mapping column names to comments for a given table.

    Parameters:
    - catalog (str): Catalog name
    - schema (str): Schema name
    - table (str): Table name

    Returns:
    - dict: { "<catalog>.<schema>.<table>": { column_name: comment, ... } }
    """
    try:
        full_table = f"{catalog}.{schema}.{table}"
        column_metadata_dict = {}
        columns_list = spark.sql(f"DESCRIBE TABLE {full_table}").collect()
        
        for row in columns_list:
            # Only include actual columns (skip metadata rows)
            if row.col_name and not row.col_name.startswith("#"):
                column_metadata_dict[row.col_name] = row.comment or ""
        
        return {full_table: column_metadata_dict}
    
    except Exception as e:
        print(f"Error getting column comments for {catalog}.{schema}.{table}: {e}")
        return {}

# Unit test
# result = get_column_comments("hive_metastore", "default", "customer")
# print(result)

In [0]:
def infer_avro_schema(df):
    """
    Infers an Avro schema from a given pandas DataFrame based on data types of the DataFrame's columns.
    Args:
        df (pandas.DataFrame): The DataFrame for which to infer the schema.
    Returns:
        dict: A dictionary representing the Avro schema with types mapped from DataFrame dtypes.
    """
    avro_types = {
        "int64": "long",
        "int32": "int",
        "float64": "double",
        "float32": "float",
        "object": "string",
        "bool": "boolean",
        "date": "string"
    }
    fields = [{"name": col, "type": avro_types.get(str(df[col].dtype), "string")} for col in df.columns]
    schema = {
        "type": "record",
        "name": "AutoGeneratedSchema",
        "fields": fields
    }
    return schema